In [ ]:
import getpass
import os

def _set_env(var:str):
	if not os.environ.get(var):
		os.environ[var] = getpass.getpass(f"{var}: ")
	
_set_env("OPENAI_API_KEY")

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 5단락 노래 가사를 훌륭하게 작성하는 작사 도우미입니다."
            "사용자의 요청에 따라 최고의 가사를 작성해주세요"
            "사용자가 피드백을 제공할 경우, 이전 시도에서 개선된 수정본을 작성해 응답하세요."
        ),
        MessagesPlaceholder(variable_name = "messages"),
    ]
)
llm = ChatOpenAI(model = "gpt-5-nano")
generate = prompt | llm

In [ ]:
lyric = ""
request = HumanMessage(
    content = "개발자로서의 삶을 이겨내는 랩 가사를 작성해줘."
)
for chunk in generate.stream({"messages":[request]}):
    print(chunk.content, end = "")
    lyric += chunk.content

In [ ]:
reflection_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 가사를 채점하는 작사가입니다. 사용자가 제출한 작사에 대한 비평과 개선 사항을 작성하세요."
            "가사의 길이, 깊이, 문체, 흥미 등을 포함해 구체적이고 비판적인 개선 요청을 제공하세요.",
        ),
        MessagesPlaceholder(variable_name = "messages"),
    ]
)
reflect = reflection_prompt | llm

In [ ]:
reflection = ""
for chunk in reflect.stream({"messages":[request, HumanMessage(content=lyric)]}):
    print(chunk.content, end = "")
    reflection += chunk.content

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages: Annotated[list, add_messages]

def generation_node(state:State)->State:
    return {"messages": [generate.invoke(state["messages"])]}

def reflection_node(state:State)->State:
    cls_map = {"ai": AIMessage, "human": HumanMessage}

    translated = [state["messages"][0]]+[
        cls_map[msg.type](content=msg.content)for msg in state["messages"][1:]
    ]

    res = reflect.invoke(translated)

    return {"messages": [HumanMessage(content = res.content)]}

In [ ]:
graph builder